# Dask

Dask erfüllt zwei verschiedene Aufgaben:

1. die dynamische Aufgabenplanung wird optimiert, ähnlich wie bei [Airflow](https://airflow.apache.org/), [Luigi](https://github.com/spotify/luigi) oder [Celery](https://docs.celeryq.dev/en/stable/)
2. Arrays, Dataframes und Lists werden parallel mit dynamischem Task Scheduling ausgeführt.

## Skalierung von Laptops bis hin zu Clustern


Dask kann mit uv auf einem Laptop installiert werden und erweitert die Größe der Datensätze von *passt in den Arbeitsspeicher* zu *passt auf die Festplatte*. Dask kann jedoch auch auf einen Cluster mit Hunderten von Rechnern skaliert werden. Dask ist robust, flexibel, Data Local und hat eine geringe Latenzzeit. Weitere Informationen findet ihr in der Dokumentation zum [Distributed Scheduler](https://distributed.dask.org/en/latest/). Dieser einfache Übergang zwischen einer einzelnen Maschine und einem Cluster ermöglicht einen einfachen Start und ein Wachstum nach Bedarf.

## Dask installieren


Ihr könnt alles installieren, was für die meisten gängigen Anwendungen von Dask erforderlich ist (Arrays, Dataframes, …). Dies installiert sowohl Dask als auch Abhängigkeiten wie NumPy, Pandas, usw., die für verschiedene Arbeiten benötigt werden:

``` bash
$ uv add "dask[complete]"
```

Es können aber auch nur einzelne Subsets installiert werden:

``` bash
$ uv add "dask[array]"
$ uv add "dask[dataframe]"
$ uv add "dask[diagnostics]"
$ uv add "dask[distributed]"
```

## Vertraute Bedienung

### Dask DataFrame

… imitiert Pandas

In [1]:
import pandas as pd


df = pd.read_csv("tutorials.csv")
grouped = df.groupby("Title")
grouped.agg("mean")

,Unnamed: 0,2021-12,2022-01,2022-02
Title,,,,
Jupyter Tutorial,0.5,18103.5,20505.5,13099.0
PyViz Tutorial,2.0,4873.0,3930.0,2573.0
Python Basics,4.5,261.0,251.0,341.0


In [2]:
import dask.dataframe as dd


df = dd.read_csv("tutorials.csv")
grouped = df.groupby("Title")
grouped.agg("mean").head()

,Unnamed: 0,2021-12,2022-01,2022-02
Title,,,,
Jupyter Tutorial,0.5,18103.5,20505.5,13099.0
PyViz Tutorial,2.0,4873.0,3930.0,2573.0
Python Basics,4.5,261.0,251.0,341.0


<div class="alert alert-block alert-info">

**Siehe auch**

* [Dask DataFrame Docs](https://docs.dask.org/en/latest/dataframe.html)
* [Dask DataFrame Best Practices](https://docs.dask.org/en/latest/dataframe-best-practices.html)
</div>

### Dask Array

… imitiert NumPy

In [3]:
import h5py
import numpy as np


f = h5py.File("mydata.h5")
x = np.array(f["."])

In [4]:
import dask.array as da


f = h5py.File("mydata.h5")
x = da.array(f["."])

<div class="alert alert-block alert-info">

**Siehe auch**

* [Dask Array Docs](https://docs.dask.org/en/latest/array.html)
* [Dask Array Best Practices](https://docs.dask.org/en/latest/array-best-practices.html)
</div>

### Dask Bag

… imitiert [iterators](https://docs.python.org/3/library/itertools.html), [Toolz](https://toolz.readthedocs.io/en/latest/index.html) und [PySpark](https://spark.apache.org/docs/latest/api/python/).

In [5]:
import dask.bag as db


b = db.from_sequence([10, 3, 5, 7, 11, 4])
list(b.topk(2))

[11, 10]

<div class="alert alert-block alert-info">

**Siehe auch**

* [Dask Bag Docs](https://docs.dask.org/en/latest/bag.html)
</div>

### Dask Delayed

… imitiert loops und umschließt benutzerdefinierten Code, siehe auch [Erstellen einer delayed-Pipeline](https://www.python4data.science/de/latest/clean-prep/dask-pipeline.html#5.-Erstellen-einer-delayed-Pipeline).

<div class="alert alert-block alert-info">

**Siehe auch**

* [Dask Delayed Docs](https://docs.dask.org/en/latest/delayed.html)
* [Dask Delayed Best Practices](https://docs.dask.org/en/latest/delayed-best-practices.html)
* [Dask-Pipeline-Beispiel: Tracking der Internationalen Raumstation mit Dask](../clean-prep/dask-pipeline.ipynb)
</div>

## ``concurrent.futures``

Das Interface ermöglicht die Übermittlung von selbstdefinierten Aufgaben.

<div class="alert alert-block alert-info">

**Bemerkung**

Für das folgende Beispiel muss Dask mit der `distributed`-Option installiert werden, z.B.

``` bash
$ uv add "dask[distributed]"
```
</div>

In [6]:
from dask.distributed import Client

In [7]:
client = Client()

Dadurch werden die lokalen Worker als Prozesse gestartet. Um die lokalen Worker als Threads auszuführen, könnt ihr `processes=False` als Parameter übergeben:

In [8]:
client = Client(processes=False)

/Users/veit/cusy/trn/jupyter-tutorial/uvenvs/py313/.venv/lib/python3.13/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 62320 instead
  warnings.warn(


Jetzt könnt ihr eure eigenen Aufgaben ausführen und Abhängigkeiten mithilfe der `submit`-Methode verketten:

In [9]:
from math import pi


def inc(x):
    return x + 1


def circumference(x):
    return 2 * pi * x


increments = client.submit(inc, 10)
circumferences = client.submit(circumference, increments)

In [10]:
circumferences.result()

69.11503837897544

<div class="alert alert-block alert-info">

**Siehe auch**

* [Dask Futures Docs](https://docs.dask.org/en/latest/futures.html)
* [Dask Futures Quickstart](https://distributed.dask.org/en/latest/quickstart.html)
* [Dask Futures Examples](https://examples.dask.org/futures.html)
</div>